# Week 4 · Topic 5 — Putting It Together

**Box we are building: STORE, the whole shape**

This week's tools combine into the shape almost every real program uses.

> **Already know this?** Skip to the *Topic 5 Challenge*.

**Resources**

Python documentation — [Data Structures](https://docs.python.org/3/tutorial/datastructures.html) — the official reference, if you want more than this walkthrough covers.

---
## 1. The shape

```
file (records.txt)                    a list of dicts
  R-001,87,100                          [ {"label": "R-001", "value": 87,  "limit": 100},
  R-002,120,100      ── load_records() →  {"label": "R-002", "value": 120, "limit": 100} ]
                                            └──────────────────┬──────────────────┘
                                                        one dict per record
```

Build one by hand first, before loading anything from a file.

In [ ]:
records = [
    {"label": "R-001", "value": 87.0, "limit": 100.0},
    {"label": "R-002", "value": 120.0, "limit": 100.0},
]

print(records[0]["label"])   # index into the list, then key into the dict

### Your turn
Build a list of 2 dicts by hand, each with `label`, `value` and `limit`. Loop over it and print just the label from each.

Animation — [List of Dicts](https://anna29.github.io/CST1510/list_of_dicts.html) — open this on the main screen to show a file's lines turning into a list of dicts.

In [ ]:
# Build a list of 2 dicts by hand
records = [
    {"label": "Sensor-A", "value": 85, "limit": 100},
    {"label": "Sensor-B", "value": 112, "limit": 100}
]

# Loop over the list and print each label
for record in records:
    print(record["label"])

---
## 2. Loading it from a file

Topic 3's `load_records` — the line-reading and splitting you already wrote, now
building the list up one dict at a time.

In [ ]:
def load_records(path):
    """Read label,value,limit lines from a file and return a list of dicts."""
    records = []
    with open(path) as f:
        for line in f:
            label, value, limit = line.strip().split(",")
            records.append({"label": label, "value": float(value), "limit": float(limit)})
    return records

### Your turn
Save 3 `label,value,limit` lines to a file of your own, then load them with `load_records`.

In [ ]:
# define the function to parse the csv lines into dicts
def load_records(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            # clean off the newline and cut at the commas
            line = line.strip()
            if not line:
                continue  # skip any accidental empty lines
            
            label, value, limit = line.split(",")
            # turn the numbers into floats so we can do math with them later
            records.append({
                "label": label,
                "value": float(value),
                "limit": float(limit)
            })
    return records


# 1. create my own file and throw 3 lines into it
with open("my_test_records.txt", "w") as f:
    f.write("CPU_Core_1,74.5,90.0\n")
    f.write("Disk_Usage,89.2,85.0\n")
    f.write("RAM_Usage,62.0,80.0\n")

# 2. read them back using the loader function
my_data = load_records("my_test_records.txt")

# check what came out
print(my_data)

---
## 3. Processing each record

The loop looks the same whether the list came from Topic 1's by-hand version or
Topic 3's file — that consistency is the point.

In [ ]:
for record in records:
    percent = (record["value"] / record["limit"]) * 100
    status = "OVER LIMIT" if percent >= 100 else "OK"
    print(f"{record['label']}: {status}")

### Your turn
Given a loaded `records` list, loop through it and print `OVER LIMIT` or `OK` for each.

In [1]:
# sample list of records (assuming this came from our file loader)
records = [
    {"label": "CPU_Core_1", "value": 74.5, "limit": 90.0},
    {"label": "Disk_Usage", "value": 89.2, "limit": 85.0},
    {"label": "RAM_Usage", "value": 62.0, "limit": 80.0}
]

# go through each record one by one
for r in records:
    # check if we crossed the line
    if r["value"] > r["limit"]:
        print(f"{r['label']}: OVER LIMIT")
    else:
        print(f"{r['label']}: OK")
        

CPU_Core_1: OK
Disk_Usage: OVER LIMIT
RAM_Usage: OK


---
## 4. Why this shape

This is the shape most real datasets already come in: rows, each with named
fields. Week 6's `pandas` reads a whole file into exactly this shape in one line —
what you just built by hand, `pandas` will hand you ready-made. Knowing the shape
first is what makes that useful instead of magic.

---
## Topic 5 Challenge

Write `load_records(path)`, load records from a file of your own, process each one
(`OVER LIMIT` / `WARNING` / `OK`), and print a summary line: `"N of M were over
limit"`. Wrap the load in a `try`/`except` so a missing file prints a clear
message instead of crashing — Topic 4's problem, applied here.

In [ ]:
# read the file and pack each line into a dict
def load_records(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            # skip any trailing blank lines
            if not line:
                continue
            
            # split on comma and cast numbers so math actually works
            label, value, limit = line.split(",")
            records.append({
                "label": label,
                "value": float(value),
                "limit": float(limit)
            })
    return records


# quick file setup with my own test data
with open("system_metrics.txt", "w") as f:
    f.write("CPU_Temp,88.5,85.0\n")       # over 100% -> OVER LIMIT
    f.write("Memory_Usage,75.0,80.0\n")   # ~93.7%   -> WARNING
    f.write("Disk_IO,42.0,100.0\n")       # 42%      -> OK
    f.write("GPU_Temp,92.0,90.0\n")       # over 100% -> OVER LIMIT


# try loading the file safely so missing files don't blow up the script
target_file = "system_metrics.txt"

try:
    records = load_records(target_file)
except FileNotFoundError:
    print(f"Error: couldn't find '{target_file}' - check the path and try again.")
    records = []

# only process if we actually got data back
if records:
    over_limit_count = 0

    print("--- Detailed Status ---")
    for r in records:
        # get percentage against limit
        pct = (r["value"] / r["limit"]) * 100

        # pick status based on thresholds
        if pct >= 100:
            status = "OVER LIMIT"
            over_limit_count += 1
        elif pct >= 90:
            status = "WARNING"
        else:
            status = "OK"

        print(f"{r['label']:<15} : {status} ({pct:.1f}%)")

    # wrap up with the summary
    print("-----------------------")
    print(f"{over_limit_count} of {len(records)} were over limit")

---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | a CSV of survey rows, each row a dict of answers |
| **Cyber Security** | a log file, each line a dict of `{ip, time, result}` |
| **IT** | a server list, each line a dict of `{hostname, used, total}` |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| `records["label"]` instead of `records[0]["label"]` | `TypeError` — `records` is a list, not a dict | index into the list first, then key into the dict |
| building the dict with the wrong keys each time | `KeyError` later, inconsistently | keep every record's dict the same shape |
| forgetting `load_records` needs to `return` the list | `None` where you expected records | check every function actually returns what it builds |

## Go deeper

`examples/01_list_of_dicts.py` — building the shape by hand
`examples/02_load_and_process.py` — the full pipeline, load then process

---
## Recap

| You can now | Using |
|---|---|
| build the shape by hand | a list of dicts |
| load it from a file | `load_records(path)` |
| process it consistently | `for record in records:` |
| see why the shape matters | it is what real datasets already look like |

**You now have all of Week 4's STORE.** Go and build your own in the lab.